# Colab/Local vLLM QLoRA Adapter Eval

Run the prompt-engineering eval CLI with a trained QLoRA adapter from Google Colab or a local Jupyter notebook. This notebook keeps the starter notebook unchanged and uses the same evaluation path as the shell command.

In [ ]:
!nvidia-smi

## Google Drive

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ModuleNotFoundError:
    IN_COLAB = False
    print('Not running in Colab; skipping Google Drive mount.')

## Repo Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/DarinAnthony/151B_SP26_Competition.git'
cwd = Path.cwd().resolve()

if IN_COLAB:
    REPO_DIR = Path('/content/151B_SP26_Competition')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif (cwd / 'pyproject.toml').exists():
    REPO_DIR = cwd
elif (cwd.parent / 'pyproject.toml').exists():
    REPO_DIR = cwd.parent
else:
    raise RuntimeError(f'Could not find repo root from {cwd}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo:', REPO_DIR)
print('Python:', sys.executable)

## Dependencies

Run this once if the notebook kernel does not already have the repo dependencies installed.

In [ ]:
%pip install -q "hydra-core>=1.3" peft bitsandbytes datasets accelerate transformers tqdm sympy pyyaml wandb

## Adapter and Run Config

In [ ]:
from pathlib import Path
import os

ADAPTER_PATH = Path(os.environ.get(
    'ADAPTER_PATH',
    '/content/drive/MyDrive/qwen_math_comp/outputs/qwen3_4b_numina_5k_smoketest/final_adapter' if IN_COLAB else '/cephfs/qwen_math_comp/outputs/qwen3_4b_numina_5k_smoketest/final_adapter',
))
RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else '/cephfs/qwen_math_comp/eval_results',
))
# Backup target used after each eval run. In Colab this defaults to Google Drive.
DRIVE_RESULTS_DIR = Path(os.environ.get(
    'DRIVE_RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else str(RESULTS_DIR),
))
RUN_NAME = os.environ.get('RUN_NAME', 'sft_numina5k_sc_terse_4096_vllm_4090')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '4096'))
RUNNER_MAX_MODEL_LEN = int(os.environ.get('RUNNER_MAX_MODEL_LEN', str(max(16384, MAX_TOKENS + 4096))))
PRIVATE_BATCH_SIZE = int(os.environ.get('PRIVATE_BATCH_SIZE', '25'))
PRIVATE_DATA_PATH = Path(os.environ.get(
    'PRIVATE_DATA_PATH',
    '/content/drive/MyDrive/qwen_math_comp/private.jsonl' if IN_COLAB else 'data/private.jsonl',
))
SUBMISSION_DIR = Path(os.environ.get(
    'SUBMISSION_DIR',
    f'/content/drive/MyDrive/qwen_math_comp/submissions/{RUN_NAME}' if IN_COLAB else str(RESULTS_DIR / 'submissions' / RUN_NAME),
))
SUBMISSION_PATH = Path(os.environ.get('SUBMISSION_PATH', str(SUBMISSION_DIR / 'private.jsonl')))

# Used only if vLLM falls back to HF/BnB. Increase this only after the smoke
# test has enough memory headroom.
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '2')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')

required_adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
missing = [name for name in required_adapter_files if not (ADAPTER_PATH / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing adapter files under {ADAPTER_PATH}: {missing}')

os.environ['ADAPTER_PATH'] = str(ADAPTER_PATH)
os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES

print('ADAPTER_PATH =', os.environ['ADAPTER_PATH'])
print('RESULTS_DIR =', RESULTS_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('WANDB_NAME =', os.environ['WANDB_NAME'])
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MAX_MODEL_LEN =', os.environ['RUNNER_MAX_MODEL_LEN'])
print('PRIVATE_BATCH_SIZE =', PRIVATE_BATCH_SIZE)
print('PRIVATE_DATA_PATH =', PRIVATE_DATA_PATH)
print('SUBMISSION_PATH =', SUBMISSION_PATH)
print('RUNNER_MICRO_BATCH_SIZE =', os.environ['RUNNER_MICRO_BATCH_SIZE'])
print('RUNNER_PARALLEL_SAMPLES =', os.environ['RUNNER_PARALLEL_SAMPLES'])
print('WANDB_PROJECT =', os.environ.get('WANDB_PROJECT', '<unset; W&B disabled>'))

## Eval Helper


In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
import time

def build_eval_cmd(run_name, *, max_tokens, results_dir, slice_indices=None):
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=sc_terse_only',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        f'runner.adapter_path={ADAPTER_PATH}',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(4, f'eval.slice_indices=[{joined}]')
    return cmd

# Copies RESULTS_DIR/<run_name> to DRIVE_RESULTS_DIR/<run_name> after every eval.
def backup_run_to_drive(run_name, *, results_dir=RESULTS_DIR, drive_results_dir=DRIVE_RESULTS_DIR):
    src = Path(results_dir) / run_name
    dst = Path(drive_results_dir) / run_name
    if not src.exists():
        print(f'No run directory to back up yet: {src}')
        return
    if src.resolve() == dst.resolve():
        print(f'Results are already on Drive: {src}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Backed up results to Google Drive: {dst}')

def run_eval(run_name, *, max_tokens=MAX_TOKENS, results_dir=RESULTS_DIR, slice_indices=None, micro_batch_size=None):
    env = os.environ.copy()
    env['ADAPTER_PATH'] = str(ADAPTER_PATH)
    env['WANDB_NAME'] = WANDB_NAME
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
    # The eval script mirrors metrics here during the run. The final backup
    # notebook cell can copy the whole run folder on demand.
    env['EVAL_METRICS_DIR'] = str(DRIVE_RESULTS_DIR)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
    env['RUNNER_MICRO_BATCH_SIZE'] = str(micro_batch_size or RUNNER_MICRO_BATCH_SIZE)

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        results_dir=results_dir,
        slice_indices=slice_indices,
    )

    print('ADAPTER_PATH =', env['ADAPTER_PATH'])
    print('WANDB_NAME =', env['WANDB_NAME'])
    print('DRIVE_RESULTS_DIR =', env['EVAL_METRICS_DIR'])
    print('RUNNER_MAX_MODEL_LEN =', env['RUNNER_MAX_MODEL_LEN'])
    print('RUNNER_MICRO_BATCH_SIZE =', env['RUNNER_MICRO_BATCH_SIZE'])
    print('RUNNER_PARALLEL_SAMPLES =', env['RUNNER_PARALLEL_SAMPLES'])
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))

    start = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

## 5-Item Smoke Test

In [ ]:
run_eval(
    f'{RUN_NAME}_smoke5',
    max_tokens=MAX_TOKENS,
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Standard Eval

In [ ]:
run_eval(
    RUN_NAME,
    max_tokens=MAX_TOKENS,
)

## Result Files

In [ ]:
from pathlib import Path

results_root = Path(RESULTS_DIR)
if not results_root.exists():
    print(f'No results directory yet: {results_root}')
else:
    for path in sorted(results_root.glob('*'))[-10:]:
        print(path)
        for metrics_name in ['metrics.json', 'leaderboard.csv', 'leaderboard.jsonl']:
            metrics_path = path / metrics_name
            if metrics_path.exists():
                print('  ', metrics_path)

## Private Submission Generation

This path uses the adapter with `sc_terse` self-consistency and does not score answers. It writes the competition submission shape: one JSON object per line with `id`, `is_mcq`, and `response`.

In [ ]:
from shared.io import load_jsonl, save_jsonl
from shared.prompt_format import build_chat_messages
from shared.runner import load_model
from shared.schemas import RunnerCfg, SamplingCfg
from shared.scoring import score_one
from shared.voting import vote_index
from experiments.prompt_engineering.src.prompts import PROMPTS

def generate_private_submission(
    *,
    private_path=PRIVATE_DATA_PATH,
    output_path=SUBMISSION_PATH,
    max_tokens=MAX_TOKENS,
    batch_size=PRIVATE_BATCH_SIZE,
    slice_indices=None,
    resume=True,
):
    private_path = Path(private_path)
    output_path = Path(output_path)
    if not private_path.exists():
        raise FileNotFoundError(
            f'Private data not found: {private_path}. Set PRIVATE_DATA_PATH to the uploaded private JSONL.'
        )
    if private_path.resolve() == output_path.resolve():
        raise ValueError('Refusing to overwrite the private input file. Choose a different SUBMISSION_PATH.')

    items = load_jsonl(private_path)
    if slice_indices is not None:
        items = [items[i] for i in slice_indices if 0 <= i < len(items)]
    if not items:
        raise ValueError(f'No private items loaded from {private_path}.')
    batch_size = max(1, int(batch_size))

    completed = {}
    if resume and output_path.exists():
        for row in load_jsonl(output_path):
            if 'id' in row and 'response' in row:
                completed[int(row['id'])] = row
        print(f'Resuming from {output_path}: {len(completed)} completed rows found.')

    prompt = PROMPTS['sc_terse']
    sampling = SamplingCfg(
        name='high_div_8',
        temperature=0.8,
        top_p=0.95,
        top_k=40,
        n_samples=8,
        repetition_penalty=1.0,
        min_p=0.0,
    )
    runner_cfg = RunnerCfg(engine='vllm', quant='bf16', adapter_path=str(ADAPTER_PATH))

    print(f'Loaded {len(items)} private items from {private_path}')
    global PRIVATE_HANDLE
    if 'PRIVATE_HANDLE' not in globals():
        print(f'Loading model with adapter: {ADAPTER_PATH}')
        PRIVATE_HANDLE = load_model(runner_cfg)
        print('Model loaded.')
    else:
        print('Reusing already-loaded adapter model.')
    pending = [item for item in items if int(item['id']) not in completed]
    print(f'Generating private responses with high_div_8 in chunks of {batch_size}: {len(pending)} pending / {len(items)} total')

    total_chunks = (len(pending) + batch_size - 1) // batch_size
    for chunk_idx, start in enumerate(range(0, len(pending), batch_size), start=1):
        chunk = pending[start:start + batch_size]
        print(f'Chunk {chunk_idx}/{total_chunks}: generating {len(chunk)} items x {sampling.n_samples} samples...')
        messages = [build_chat_messages(item, prompt) for item in chunk]
        outputs = PRIVATE_HANDLE.generate_batch(messages, sampling, max_tokens)
        for item, out in zip(chunk, outputs):
            dummy_item = dict(item)
            dummy_item['answer'] = 'A' if item.get('options') else '0'
            extracts = [score_one(dummy_item, response).extracted for response in out.responses]
            selected_idx = vote_index(extracts)
            completed[int(item['id'])] = {
                'id': int(item['id']),
                'is_mcq': bool(item.get('options')),
                'response': out.responses[selected_idx],
            }
        rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
        save_jsonl(rows, output_path)
        print(f'  saved {len(rows)}/{len(items)} rows to {output_path}')

    rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
    save_jsonl(rows, output_path)
    print(f'Saved {len(rows)} private submission rows to {output_path}')
    return output_path

## 5-Item Private Generation Smoke

In [ ]:
generate_private_submission(
    output_path=SUBMISSION_PATH.with_name('private_smoke5.jsonl'),
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Private Submission

In [ ]:
# Run this after the private smoke file looks good.
generate_private_submission()

## Back Up Results to Google Drive


In [ ]:
# Run this after a smoke or full eval to copy that run's output folder to Google Drive.
# The most recent smoke run is f'{RUN_NAME}_smoke5'; the full run is RUN_NAME.
backup_run_to_drive(f'{RUN_NAME}_smoke5')
backup_run_to_drive(RUN_NAME)
